
# Validation run

This runs sql/01_cleaning_and_checks.sql and sql/02_analysis.sql against
the data package and shows what each query returns. The write-up of what
these results mean, and the reasoning behind each step, is in
../INVESTIGATION.md.


In [1]:

import duckdb
from pathlib import Path

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'sql').exists() else _cwd.parent

con = duckdb.connect(':memory:')

def run_file(path):
    sql_text = (ROOT / 'sql' / path).read_text().replace("'data/", f"'{ROOT}/data/")
    stmts = [s.strip() for s in '\n'.join(
        l for l in sql_text.splitlines() if not l.strip().startswith('--')
    ).split(';') if s.strip()]
    return [con.execute(s).df() for s in stmts]


## sql/01_cleaning_and_checks.sql

In [2]:

cleaning_results = run_file('01_cleaning_and_checks.sql')
for df in cleaning_results:
    if not df.empty:
        print(df.to_string(index=False))
        print()


 initial_rows  incremental_distinct_ids  incremental_rows_including_dupes  overlap_trip_ids
       965201                    305357                            305457                50

 conflicting_rows
                0

earliest_late_arrival  n_late_arrivals
  2025-03-03 09:00:00              125

 pickup_community_area  n_rows
                   999      25

 n_rows  null_trip_total  pct_null_trip_total
1270483           3777.0                  0.3

community_area_name      n  pct_missing_dropoff
        SOUTH SHORE 118644                  8.4
     GARFIELD RIDGE 156659                 31.9
          HYDE PARK 211727                  2.6
          WEST TOWN 478171                  3.9

 trips_over_100_pct_time  worst_value
                106728.0        3.793

 mismatched_totals  matched_without_authorization
                 0                              0

dedup_check  n_rows  n_distinct_ids
       pass 1270483         1270483

reconciliation_check  actual_trusted_rows  expected

## sql/02_analysis.sql

In [3]:

analysis_results = run_file('02_analysis.sql')
for df in analysis_results:
    if not df.empty:
        print(df.to_string(index=False))
        print()


 max_count_diff  max_price_diff
              0             0.0

 min_pct_off  max_pct_off
         0.0     0.209792

community_area_name week_start  trip_volume  avg_trip_price  shared_trip_rate
     GARFIELD RIDGE 2025-03-03        24812           42.43             0.007
     GARFIELD RIDGE 2025-03-10        27180           43.55             0.007
     GARFIELD RIDGE 2025-03-17        24763           42.12             0.009
     GARFIELD RIDGE 2025-03-24        26152           43.70             0.007
     GARFIELD RIDGE 2025-03-31        28025           45.34             0.008
     GARFIELD RIDGE 2025-04-07        25742           41.34             0.007
     GARFIELD RIDGE 2025-04-14        24863           41.37             0.007
     GARFIELD RIDGE 2025-04-21        28133           43.87             0.007
          HYDE PARK 2025-03-03        43607           13.53             0.280
          HYDE PARK 2025-03-10        36085           18.76             0.220
          HYDE PARK 2025